In [39]:
from pathlib import Path
import json
import re
import subprocess
import pandas as pd
from joblib import Parallel, delayed
import shutil

BASE = Path("/home/rph/scalesurfer/data")
ADNI_ROOT = BASE / "ADNI"
BIDS_ROOT = BASE / "adni_bids"
CONFIG = BASE / "config_adni.json"

T1_KEYWORDS = (
    "MPRAGE",
    "MPR",
    "SPGR",
    "FSPGR",
    "IRSPGR",
    "IRFSPGR",
    "T1",
)

def norm_seq(s: str) -> str:
    return re.sub(r"[^A-Z0-9]+", "", s.upper())

def is_t1_sequence(seq: str) -> bool:
    s = norm_seq(seq)
    return any(k in s for k in T1_KEYWORDS)

def bids_label(s: str) -> str:
    """BIDS labels should not contain underscores, dashes, dots, etc."""
    return re.sub(r"[^A-Za-z0-9]", "", s)

def date_to_session(date_dir: str) -> str:
    """2013-03-20_12_07_11.0 -> 20130320"""
    return bids_label(date_dir.split("_")[0])

def bids_t1_exists(subj_bids: str, ses_bids: str) -> bool:
    anat_dir = BIDS_ROOT / f"sub-{subj_bids}" / f"ses-{ses_bids}" / "anat"
    return any(anat_dir.glob(f"sub-{subj_bids}_ses-{ses_bids}*_T1w.nii.gz"))

def make_config():
    CONFIG.parent.mkdir(parents=True, exist_ok=True)
    cfg = {
        "descriptions": [
            {
                "datatype": "anat",
                "suffix": "T1w",
                "criteria": {
                    "SeriesDescription": "*"
                },
            }
        ]
    }
    CONFIG.write_text(json.dumps(cfg, indent=2))


def collect_series():
    rows = []

    for img_dir in ADNI_ROOT.glob("*/*/*/I*"):
        if not img_dir.is_dir():
            continue

        dcm_files = list(img_dir.glob("*.dcm"))
        if not dcm_files:
            continue

        subj = img_dir.parts[-4]
        seq = img_dir.parts[-3]
        date = img_dir.parts[-2]
        image_id = img_dir.parts[-1]

        if not is_t1_sequence(seq):
            continue

        rows.append(
            {
                "subject_adni": subj,
                "subject_bids": bids_label(subj),
                "sequence": seq,
                "date": date,
                "session_bids": date_to_session(date),
                "image_id": image_id,
                "dicom_dir": str(img_dir),
                "n_dicoms": len(dcm_files),
            }
        )

    return pd.DataFrame(rows).sort_values(
        ["subject_adni", "session_bids", "sequence", "image_id"]
    )

def run_one_dcm2bids_job(subj_bids, ses_bids, dicom_dirs, work_root):
    work_dir = work_root / f"sub-{subj_bids}_ses-{ses_bids}"

    if work_dir.exists():
        shutil.rmtree(work_dir)

    work_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "dcm2bids",
        "-d",
        *dicom_dirs,
        "-p",
        subj_bids,
        "-s",
        ses_bids,
        "-c",
        str(CONFIG),
        "-o",
        str(work_dir),
        "--force_dcm2bids",
        "--clobber",
    ]

    result = subprocess.run(
        cmd,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    return {
        "subject": subj_bids,
        "session": ses_bids,
        "work_dir": str(work_dir),
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }

def merge_bids_outputs(results, bids_root):
    for r in results:
        if r["returncode"] != 0:
            continue

        work_dir = Path(r["work_dir"])
        for sub_dir in work_dir.glob("sub-*"):
            dst = bids_root / sub_dir.name
            shutil.copytree(sub_dir, dst, dirs_exist_ok=True)


def run_dcm2bids(df: pd.DataFrame, n_jobs: int = 16, skip_existing: bool = True):
    BIDS_ROOT.mkdir(parents=True, exist_ok=True)

    work_root = BIDS_ROOT / "_dcm2bids_parallel_work"
    work_root.mkdir(parents=True, exist_ok=True)

    jobs = []
    skipped = []

    for (subj_bids, ses_bids), g in df.groupby(["subject_bids", "session_bids"]):
        if skip_existing and bids_t1_exists(subj_bids, ses_bids):
            skipped.append((subj_bids, ses_bids))
            continue

        dicom_dirs = g["dicom_dir"].tolist()
        jobs.append((subj_bids, ses_bids, dicom_dirs))

    print(f"Skipping {len(skipped)} existing subject/session outputs")
    print(f"Running {len(jobs)} dcm2bids jobs with n_jobs={n_jobs}")

    if not jobs:
        print("Nothing to do.")
        return

    results = Parallel(n_jobs=n_jobs, prefer="threads")(
        delayed(run_one_dcm2bids_job)(subj_bids, ses_bids, dicom_dirs, work_root)
        for subj_bids, ses_bids, dicom_dirs in jobs
    )

    failed = [r for r in results if r["returncode"] != 0]

    merge_bids_outputs(results, BIDS_ROOT)

    if failed:
        fail_log = BIDS_ROOT / "dcm2bids_failed_jobs.txt"
        with open(fail_log, "w") as f:
            for r in failed:
                f.write("=" * 80 + "\n")
                f.write(f"sub-{r['subject']} ses-{r['session']}\n")
                f.write("=" * 80 + "\n")
                f.write(r["stderr"])
                f.write("\n\n")

        raise RuntimeError(f"{len(failed)} dcm2bids jobs failed. See {fail_log}")

    print("All dcm2bids jobs completed.")

In [40]:
make_config()

df = collect_series()
if df.empty:
    raise RuntimeError(f"No T1 DICOM series found under {ADNI_ROOT}")

BIDS_ROOT.mkdir(parents=True, exist_ok=True)
manifest = BIDS_ROOT / "sourcedata_adni_t1_manifest.tsv"
df.to_csv(manifest, sep="\t", index=False)

print(f"Found {len(df)} T1-ish ADNI image folders")
print(f"Wrote manifest: {manifest}")

run_dcm2bids(df, n_jobs=16, skip_existing=True)

Found 1523 T1-ish ADNI image folders
Wrote manifest: /home/rph/scalesurfer/data/adni_bids/sourcedata_adni_t1_manifest.tsv
Skipping 1414 existing subject/session outputs
Running 109 dcm2bids jobs with n_jobs=16
All dcm2bids jobs completed.
